# AI算力驱动下的光模块行业财务分析

比较中际旭创、新易盛、光迅科技 2023—2025 年财务表现。本项目使用 pandas、NumPy 与 matplotlib 完成描述性分析、规则筛查和透明的加权评分，不构成投资建议。

## 1. 项目说明
本 Notebook 从项目 `data/` 目录读取原始 Excel，在 `outputs/` 生成分析表和图表。分析仅基于原始工作簿 `Raw_Data` 中已有数据，不补造数据。

## 2. 导入库与基础配置
统一在开头导入依赖，并设置图表尺寸、中文字体候选项和输出目录。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 从项目根目录或 notebooks 目录运行都能定位数据与输出目录
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists() and (PROJECT_DIR.parent / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent
DATA_PATH = PROJECT_DIR / "data" / "光模块财务数据.xlsx"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Windows 常见中文字体优先；按系统可用字体回退
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
FIGSIZE = (9, 5.5)
DPI = 300


## 3. 读取数据
读取原始工作簿的 `Raw_Data` 工作表。
分析输入只包含七个核心原始字段；原始工作簿完整备份在 `data/original/`，Python 自行计算同比和现金转换率。


### 财务口径修订说明

本项目输入表的“毛利率”统一为发行人合并报表口径，按（合并营业收入－合并营业成本）/合并营业收入计算，保留四位小数；调整前输入备份保留。原先的分部/行业毛利率另列于项目根目录“研究底稿/财务口径核查表.xlsx”，不与合并毛利率混合评分。

新易盛ROE使用2025年年度报告第7页披露的“加权平均净资产收益率”调整后历史列：2023年5.25%、2024年21.66%、2025年72.75%。年报将追溯调整原因列为“其他原因”，具体机制无法由该披露确认。其他公司ROE同样按年报披露的加权平均净资产收益率口径。来源和逐项数据见财务口径核查表。


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="Raw_Data")
print(f"读取文件：{DATA_PATH}")
print(f"数据规模：{df.shape[0]} 行 × {df.shape[1]} 列")


## 4. 数据检查
检查列名、数据类型、缺失值、年份和公司，确认数据范围；原始字段若有疑问应回到来源核实。
必要列、公司名称、数值、整数年份和公司+年份唯一性必须通过校验；缺失、无效值、重复记录或净利润为零时明确报错，不修改或猜测原始数据。


In [ ]:
# 校验必要字段，发现问题立即报错，不补填或猜测数据。
required_columns = ["公司", "年份", "营业收入（亿元）", "归母净利润（亿元）", "毛利率", "ROE", "经营现金流（亿元）"]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"输入缺少必要列：{missing_columns}")
if df.empty:
    raise ValueError("Raw_Data 没有财务数据。")
if df["公司"].isna().any() or df["公司"].astype(str).str.strip().eq("").any():
    raise ValueError("公司列存在缺失或空白名称，请核查原始数据。")

# 校验数值和年份；仅转换可解释的数值类型，不修改财务数字。
for column in required_columns[1:]:
    numeric_values = pd.to_numeric(df[column], errors="coerce")
    invalid_values = numeric_values.isna() | ~np.isfinite(numeric_values)
    if invalid_values.any():
        excel_rows = (df.index[invalid_values] + 2).tolist()
        raise ValueError(f"列 {column} 存在缺失、无法转换或非有限数值，Excel 行：{excel_rows}")
    df[column] = numeric_values
if (df["年份"] % 1 != 0).any():
    raise ValueError("年份必须为整数，请核查原始数据。")
df["年份"] = df["年份"].astype(int)
if df.duplicated(["公司", "年份"], keep=False).any():
    duplicate_rows = df.loc[df.duplicated(["公司", "年份"], keep=False), ["公司", "年份"]]
    raise ValueError(f"公司+年份存在重复记录：\n{duplicate_rows.to_string(index=False)}")
if df["归母净利润（亿元）"].eq(0).any():
    zero_profit_rows = df.loc[df["归母净利润（亿元）"].eq(0), ["公司", "年份"]]
    raise ValueError(f"归母净利润为 0，无法计算现金转换率：\n{zero_profit_rows.to_string(index=False)}")

print("字段：", df.columns.tolist())
print("\n数据类型与非空数量：")
df.info()
print("\n缺失值数量：\n", df.isna().sum())
print("\n年份：", sorted(df["年份"].dropna().unique().tolist()))
print("公司：", df["公司"].dropna().unique().tolist())
print("\n原始数据预览：")
display(df.head())

## 5. 财务指标计算
按公司、年份排序。同比按相邻年度计算；首年没有前一年数据，因此同比留空。现金转换率定义为经营现金流/归母净利润。
毛利率与 ROE 的同比变化采用本年减上年的差值，保留比例单位：差值 0.01 对应增加 1 个百分点，不是增加 0.01 个百分点。


In [ ]:
df = df.sort_values(["公司", "年份"]).reset_index(drop=True)
grouped = df.groupby("公司", sort=False)
df["营收同比"] = grouped["营业收入（亿元）"].pct_change(fill_method=None)
df["净利润同比"] = grouped["归母净利润（亿元）"].pct_change(fill_method=None)
# 数据校验已确认归母净利润不为 0，安全计算现金转换率
df["现金转换率"] = df["经营现金流（亿元）"] / df["归母净利润（亿元）"]
# 毛利率与 ROE 的变化保留比例单位，例如 0.01 表示上升 1 个百分点
df["毛利率同比变化"] = grouped["毛利率"].diff()
df["ROE同比变化"] = grouped["ROE"].diff()

## 6. 公司汇总分析
按公司汇总营收、净利润和经营现金流的均值及各指标末年值，便于概览规模与变化。

In [ ]:
company_summary = df.groupby("公司").agg(
    营收均值_亿元=("营业收入（亿元）", "mean"),
    净利润均值_亿元=("归母净利润（亿元）", "mean"),
    经营现金流均值_亿元=("经营现金流（亿元）", "mean"),
    最新年份=("年份", "max"),
).reset_index()
latest_rows = df.sort_values("年份").groupby("公司").tail(1)[["公司", "营业收入（亿元）", "归母净利润（亿元）", "毛利率", "ROE", "经营现金流（亿元）"]]
company_summary = company_summary.merge(latest_rows, on="公司", how="left")
display(company_summary)

## 7. 最新年份横向比较
使用 `analysis_year = int(df["年份"].max())` 自动识别数据中的最大年份，筛选 `df_latest` 并生成 `comparison_latest`。保留核心指标、同比与现金转换率，按营业收入从高到低排序。


In [ ]:
analysis_year = int(df["年份"].max())
print(f"横向比较分析年份：{analysis_year}（数据中的最大年份）")
df_latest = df.loc[df["年份"] == analysis_year].copy()
comparison_latest = df_latest[["公司", "营业收入（亿元）", "归母净利润（亿元）", "毛利率", "ROE", "经营现金流（亿元）", "营收同比", "净利润同比", "现金转换率"]].sort_values("营业收入（亿元）", ascending=False).reset_index(drop=True)
display(comparison_latest)

## 8. 财务风险筛查
规则为：现金转换率 < 0.5、经营现金流 < 0、毛利率同比下降、ROE同比下降。风险信号只是“筛查提示”，不等于公司存在确定风险；阈值和变化不代表审计结论。
首年缺少上年毛利率和 ROE，不能判断是否下降，对应规则不触发；这不代表已证实不存在风险。信号数是触发规则的数量，其中现金转换率低与经营现金流为负可能同时触发，不应解释为互相独立的风险事件数量。


In [ ]:
df["风险_现金转换率低"] = df["现金转换率"] < 0.5
df["风险_经营现金流为负"] = df["经营现金流（亿元）"] < 0
df["风险_毛利率下降"] = df["毛利率同比变化"] < 0
df["风险_ROE下降"] = df["ROE同比变化"] < 0
risk_flags = ["风险_现金转换率低", "风险_经营现金流为负", "风险_毛利率下降", "风险_ROE下降"]
df["风险信号数"] = df[risk_flags].sum(axis=1)
risk_summary = df.groupby("公司")[risk_flags + ["风险信号数"]].sum().reset_index()
display(risk_summary)

## 9. 风险年份识别
列出至少触发一项筛查规则的公司年份，并保留信号数。

In [ ]:
risk_years = df.loc[df["风险信号数"] > 0].copy()
display(risk_years[["公司", "年份"] + risk_flags + ["风险信号数"]])

## 10. 财务趋势可视化
三项指标分别绘图，采用统一尺寸、标题、坐标轴和图例，并以 300 dpi 保存 PNG。年份横坐标从数据中自动取得。
ROE 图表使用 `ROE * 100`，纵轴单位为百分数；数据表和评分输入的 ROE 仍保留原来的比例单位。


In [ ]:
years = sorted(df["年份"].dropna().unique())
plot_specs = [("营业收入（亿元）", "营业收入趋势", "营业收入（亿元）", "营业收入趋势.png"), ("归母净利润（亿元）", "归母净利润趋势", "归母净利润（亿元）", "归母净利润趋势.png"), ("ROE", "ROE趋势", "ROE（%）", "ROE趋势.png")]
for column, title, ylabel, filename in plot_specs:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    for company, company_df in df.groupby("公司"):
        # 仅图表将 ROE 转为百分数，原始指标与评分仍使用比例值
        plot_values = company_df[column] * 100 if column == "ROE" else company_df[column]
        ax.plot(company_df["年份"], plot_values, marker="o", linewidth=2, label=company)
    ax.set_title(title)
    ax.set_xlabel("年份")
    ax.set_ylabel(ylabel)
    ax.set_xticks(years)
    ax.legend(title="公司")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 11. NumPy标准化
标准化让营收同比、ROE、毛利率和现金转换率等不同量纲的指标可比较。Z-score = (数值−均值)/标准差，表示某值离均值有多少个标准差。`axis=0` 表示沿行方向汇总、对每一列分别求统计量。若标准差为 0，则该列无法区分公司，Z-score 设为 0。
本项目把这三家公司的自动识别最新年份数据作为比较集合，使用 `np.std(..., ddof=0)`（总体标准差）；与 pandas 默认 `std()` 的 `ddof=1` 口径不同。标准化仅使用 `analysis_year` 年横向数据。


In [ ]:
score_data = comparison_latest[["公司", "营收同比", "ROE", "毛利率", "现金转换率"]].copy()
feature_columns = ["营收同比", "ROE", "毛利率", "现金转换率"]
values = score_data[feature_columns].to_numpy(dtype=float)
if not np.isfinite(values).all():
    raise ValueError("最新年份评分指标存在缺失或非有限值，请核查同比计算和原始数据。")
feature_means = np.mean(values, axis=0)
feature_stds = np.std(values, axis=0, ddof=0)
# 防止某项指标在样本内完全相同时发生除以 0
safe_stds = np.where(feature_stds == 0, 1, feature_stds)
z_values = (values - feature_means) / safe_stds
z_values[:, feature_stds == 0] = 0
z_columns = ["营收同比_Z", "ROE_Z", "毛利率_Z", "现金转换率_Z"]
score_data[z_columns] = z_values
print("均值：", feature_means)
print("标准差：", feature_stds)
display(score_data)

## 12. 综合评分模型
`np.dot` 在这里将每家公司四项 Z-score 与一组权重相乘并求和，得到加权综合分。分数仅用于这三家公司在选定年份内的相对比较。
评分前检查每套权重的合计是否为 1；三套权重的指标顺序依次为营收同比、ROE、毛利率、现金转换率。


In [ ]:
weight_schemes = {
    "等权": np.array([0.25, 0.25, 0.25, 0.25]),
    "成长盈利型": np.array([0.30, 0.30, 0.20, 0.20]),
    "质量现金型": np.array([0.20, 0.25, 0.25, 0.30]),
}
z_matrix = score_data[z_columns].to_numpy()
for scheme_name, weights in weight_schemes.items():
    # 校验权重合计为 1，再使用既有加权评分方法
    if not np.isclose(np.sum(weights), 1.0, rtol=0, atol=1e-10):
        raise ValueError(f"{scheme_name}的权重加总不为 1：{np.sum(weights)}")
    score_data[f"{scheme_name}得分"] = np.dot(z_matrix, weights)
    score_data[f"{scheme_name}排名"] = score_data[f"{scheme_name}得分"].rank(ascending=False, method="min").astype(int)
display(score_data[["公司"] + [f"{name}{suffix}" for name in weight_schemes for suffix in ("得分", "排名")]])

## 13. 权重敏感性分析
权重代表分析者对指标重要性的设定，不是客观真理。比较三套事先说明的权重方案，可以观察名次是否稳定；如果排名变化，结论对权重设定较敏感，应谨慎解释。
这里仅比较已设定的三套权重。三套方案排名一致，只能说明结论在这些方案下保持一致，不能说明对所有可能的权重都不敏感。


In [ ]:
score_model = score_data.copy()
rank_columns = [f"{name}排名" for name in weight_schemes]
score_model["排名最小值"] = score_model[rank_columns].min(axis=1)
score_model["排名最大值"] = score_model[rank_columns].max(axis=1)
score_model["排名变动范围"] = score_model["排名最大值"] - score_model["排名最小值"]
score_model = score_model.sort_values("等权排名").reset_index(drop=True)
display(score_model)

## 14. 结果导出
导出处理后数据、已有公司汇总 `Company_Summary`、最新年份横向比较、公司风险汇总、风险年份与评分模型。比较工作表命名为 `Comparison_{analysis_year}`，当前数据自动识别为 2025 年，因此导出为 `Comparison_2025`；该 Sheet 名不决定分析年份。Excel 不写入 Pandas 行索引。


In [ ]:
excel_path = OUTPUT_DIR / "光模块_Python分析结果.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Processed_Data", index=False)
    company_summary.to_excel(writer, sheet_name="Company_Summary", index=False)
    comparison_latest.to_excel(writer, sheet_name=f"Comparison_{analysis_year}", index=False)
    risk_summary.to_excel(writer, sheet_name="Risk_Summary", index=False)
    risk_years.to_excel(writer, sheet_name="Risk_Years", index=False)
    score_model.to_excel(writer, sheet_name="Score_Model", index=False)
print(f"已导出：{excel_path}")

## 15. 研究结论与模型局限
在给定 2023—2025 数据中，三家公司最新年份的规模、成长、盈利能力和现金转换表现存在差异；风险规则标记的是需要进一步核查的年份，评分排名也只是所选指标与权重下的横向结果。

局限：样本仅三家公司、三年；数据依赖原始 Excel，未独立核对公告口径；未考虑估值、业务结构、客户集中度、资本开支、行业周期等因素；权重具有主观性，Z-score 随样本变化。风险提示和综合分均不可单独用于投资决策。
本评分模型为简化的横向研究辅助工具。当前将现金转换率作为正向指标处理，但现金转换率并非在所有情况下越高越好，需要结合营运资本变化、回款节奏和业务背景解释。
